Автор: Вебер Матвей


In [ ]:
%pip install -U lyceum-prg-2627==0.4.0

# Pandas. Рабочие приёмы

Те же четыре темы, что в первой тетрадке: взгляд на таблицу, обращение к строкам, группировка, среднее с максимумом и минимумом. Здесь каждый приём доводится до рабочей формы.

**Как работать**

1. Запусти установку пакета и ячейку импорта.
2. В каждой задаче замени `pass` на решение с `return`.
3. Обычная задача проверяется через `pandas02.check(...)`, звёздочка отдельно через `pandas02.check_star(...)`.

Проверка вызывает функцию на скрытых небольших таблицах. Сами таблицы и эталонные ответы не показываются.

In [ ]:
from lyceum_prg_2627.data_analysis import pandas02
import pandas as pd

---
## 1. Типы, пропуски и сводка

- `df.dtypes` показывает тип каждого столбца;
- `df.info()` показывает типы и число непропущенных значений;
- `df.describe()` для числового столбца даёт сводку, поле `count` в ней считает строки без пропусков;
- `df[col].isna().sum()` считает пропуски;
- `mean` пропускает `NaN` и делит сумму на число оставшихся значений.

In [ ]:
scores = pd.DataFrame({"балл": [1.0, None, 3.0, 5.0]})
scores.dtypes

In [ ]:
scores.info()

In [ ]:
scores.describe()

In [ ]:
print(scores["балл"].isna().sum())
print(scores["балл"].mean())

---
Напиши функцию `task01(df, col)`.

Верни кортеж:

1. `str(df[col].dtype)`;
2. число пропусков в столбце;
3. среднее столбца;
4. поле `count` из `df[col].describe()`.

In [ ]:
def task01(df, col):
    # твой код
    pass

In [ ]:
pandas02.check(1, task01)

---
## 1*. Числа, записанные текстом

Звёздочка. Столбец может выглядеть как числа, но `dtypes` называет его текстом. Тогда `mean` не считает обычное среднее.

In [ ]:
kinds = pd.DataFrame({
    "имя": ["Аня", "Боря", "Вика"],
    "балл_текст": ["4", "5", "3"],
    "балл": [4, 5, 3],
})
print(kinds.dtypes)
print(kinds["балл"].mean())
try:
    print(kinds["балл_текст"].mean())
except TypeError:
    print("TypeError: по текстовому столбцу среднее не считается")

---
Напиши функцию `task01_star(df)`.

Найди столбец, который выглядит как числа, но имеет текстовый dtype. Верни кортеж: имя столбца и `str(dtype)`. Имя и dtype бери из таблицы, не подставляй их вручную.

In [ ]:
def task01_star(df):
    # твой код
    pass

In [ ]:
pandas02.check_star(1, task01_star)

---
## 2. Срезы, несколько условий, запись в ячейку

У обычного индекса `loc[start:end]` включает правый конец, `iloc[start:end]` не включает.

Несколько условий соединяй через `&` и `|`. Каждое сравнение бери в скобки. У `&` приоритет выше, чем у `|`, поэтому скобки вокруг `|` меняют смысл.

Запись в одну ячейку: `df.loc[label, col] = new_value`.

In [ ]:
nums = pd.Series([10, 20, 30, 40])
print(nums.loc[1:3].tolist())
print(nums.iloc[1:3].tolist())

In [ ]:
visits = pd.DataFrame({
    "очки": [5, 15, 25, 35],
    "возраст": [14, 19, 16, 11],
    "город": ["Тула", "Тула", "Омск", "Тула"],
    "кружок": ["хор", "роботы", "хор", "роботы"],
})
age = visits["возраст"] >= 16
city = visits["город"] == "Тула"
club = visits["кружок"] == "хор"
print(int((age & city).sum()))
print(int((city | club).sum()))
print(int((age & (city | club)).sum()))

In [ ]:
fixed = visits.copy()
fixed.loc[0, "кружок"] = "шахматы"
fixed.head(1)

---
Напиши функцию `task02(df, col, start, end, mask_col, low, flag_col, flag_value, extra_col, extra_value)`.

Верни кортеж:

1. список `df[col].loc[start:end]`;
2. список `df[col].iloc[start:end]`;
3. число строк, где `(df[mask_col] >= low) & (df[flag_col] == flag_value)`;
4. число строк, где `(df[flag_col] == flag_value) | (df[extra_col] == extra_value)`;
5. число строк, где `(df[mask_col] >= low) & ((df[flag_col] == flag_value) | (df[extra_col] == extra_value))`.

В пятом пункте внутренние скобки обязательны.

In [ ]:
def task02(df, col, start, end, mask_col, low, flag_col, flag_value, extra_col, extra_value):
    # твой код
    pass

In [ ]:
pandas02.check(2, task02)

---
## 2*. Интервал, два столбца, одна замена

Звёздочка. Напиши функцию `task02_star(df, value_col, low, high, other_col, fix_label, new_value)`.

1. Оставь строки, где `low <= df[value_col] <= high`.
2. Оставь два столбца: сначала `value_col`, потом `other_col`.
3. После фильтра запиши `new_value` в `other_col` по метке `fix_label` через `loc`.
4. Верни эту таблицу.

In [ ]:
def task02_star(df, value_col, low, high, other_col, fix_label, new_value):
    # твой код
    pass

In [ ]:
pandas02.check_star(2, task02_star)

---
## 3. Две группы и три агрегата в одной таблице

`groupby` по двум столбцам с `as_index=False` оставляет обычную таблицу, а не серию с мультииндексом.

`agg` собирает среднее, максимум и минимум в соседние столбцы. Дальше таблицу сортируют и сбрасывают индекс.

In [ ]:
cities = pd.DataFrame({
    "город": ["Тула", "Омск", "Тула", "Омск"],
    "смена": ["утро", "утро", "вечер", "вечер"],
    "балл": [3, 7, 5, 1],
})
grouped = cities.groupby(["город", "смена"], as_index=False).agg(
    mean=("балл", "mean"),
    max=("балл", "max"),
    min=("балл", "min"),
)
grouped.sort_values(["город", "смена"]).reset_index(drop=True)

---
Напиши функцию `task03(df, key_a, key_b, value_col)`.

Сгруппируй по `key_a` и `key_b` с `as_index=False`. Посчитай `mean`, `max` и `min` столбца `value_col` под этими именами. Отсортируй строки по `key_a`, затем по `key_b`, и сбрось индекс.

In [ ]:
def task03(df, key_a, key_b, value_col):
    # твой код
    pass

In [ ]:
pandas02.check(3, task03)

---
## 3*. Среднее и максимум, порог по максимуму

Звёздочка. Напиши функцию `task03_star(df, group_col, value_col, threshold)`.

Для каждой группы посчитай `mean` и `max`. Оставь группы, у которых `max` строго больше `threshold`. Верни таблицу со столбцами группы, `mean` и `max`. Результат отсортируй по группе. `as_index=False` нужен, чтобы это осталось таблицей.

In [ ]:
def task03_star(df, group_col, value_col, threshold):
    # твой код
    pass

In [ ]:
pandas02.check_star(3, task03_star)

---
## 4. Строка максимума и пропуск внутри группы

`idxmax` возвращает метку строки, в которой лежит максимум, а не сам максимум. Сам максимум даёт `max`.

Пропуск не удаляет группу: `size` или `len` считают и пустую ячейку. `mean` эту ячейку пропускает, поэтому в среднее входит меньше значений, чем строк в группе.

In [ ]:
marks = pd.DataFrame({
    "группа": ["А", "А", "Б", "Б"],
    "очки": [2.0, None, 6.0, 1.0],
})
group = marks.loc[marks["группа"] == "А", "очки"]
print(marks["очки"].idxmax())
print(marks["очки"].max())
print(len(group))
print(group.mean())
print(int(group.count()))

---
Напиши функцию `task04(df, group_col, value_col, group_name)`.

Верни кортеж:

1. `df[value_col].idxmax()`;
2. `df[value_col].max()`;
3. число строк группы `group_name`, вместе с пропусками;
4. среднее `value_col` в этой группе;
5. число непропущенных значений, которые вошли в это среднее.

In [ ]:
def task04(df, group_col, value_col, group_name):
    # твой код
    pass

In [ ]:
pandas02.check(4, task04)

---
## 4*. Вся строка, где значение максимально

Звёздочка. Напиши функцию `task04_star(df, group_col, value_col)`.

Для каждой группы верни строку целиком, в которой `value_col` достигает максимума. Если таких строк несколько, оставь первую, как это делает `idxmax`. Результат отсортируй по `group_col`.

In [ ]:
def task04_star(df, group_col, value_col):
    # твой код
    pass

In [ ]:
pandas02.check_star(4, task04_star)